In [ ]:

members = {
    "M100": {
        "name": "John Smith",
        "membership_type": "Standard",
        "status": "Active",
        "branch": "CEN",
        "outstanding_fines": 0,
        "active_loans": []
    },

    "M101": {
        "name": "Mary Jones",
        "membership_type": "Senior",
        "status": "Active",
        "branch": "NOR",
        "outstanding_fines": 5,
        "active_loans": []
    }
}

branches = {
    "CEN": {
        "name": "Central Library",
        "max_daily_loans": 50,
        "active_loans": 10
    },

    "NOR": {
        "name": "North Library",
        "max_daily_loans": 40,
        "active_loans": 40
    }
}

staff_accounts = {
    "admin1": {
        "password": "admin123",
        "role": "Administrator",
        "full_name": "Alice Johnson"
    },

    "lib1": {
        "password": "lib123",
        "role": "Librarian",
        "full_name": "Brian Daniels"
    }
}

loan_records = []

def junior_fine(days_overdue, first_offence):
    

    if first_offence:
        return 0.0

    return days_overdue * 1.00


def standard_fine(days_overdue, num_books):
    

    return days_overdue * 2.50 * num_books


def senior_fine(days_overdue, num_books):
   

    fine_per_book = days_overdue * 1.00

    if fine_per_book > 30:
        fine_per_book = 30

    return fine_per_book * num_books


def student_fine(days_overdue, num_books):
    

    return days_overdue * 1.50 * num_books

def calculate_fine(membership_type,
                   days_overdue,
                   num_books,
                   first_offence=False):
    

    if days_overdue <= 0 or num_books <= 0:
        return 0.0

    membership_type = membership_type.lower()

    if membership_type == "junior":
        return junior_fine(days_overdue,
                           first_offence) * num_books

    elif membership_type == "standard":
        return standard_fine(days_overdue,
                             num_books)

    elif membership_type == "senior":
        return senior_fine(days_overdue,
                           num_books)

    elif membership_type == "student":
        return student_fine(days_overdue,
                            num_books)

    else:
        return 0.0

def issue_loan(loan_records,
               members,
               branches,
               member_id,
               isbn,
               issue_date):
    

    
    if member_id not in members:
        print("Member not found!")
        return None

    member = members[member_id]

    if member["status"] != "Active":
        print("Member account inactive!")
        return None

    branch_code = member["branch"]

    
    if branch_code not in branches:
        print("Branch not found!")
        return None

    branch = branches[branch_code]

  
    if branch["active_loans"] >= branch["max_daily_loans"]:
        print("Branch loan capacity reached!")
        return None

    if len(isbn) != 13 or not isbn.isdigit():
        print("Invalid ISBN!")
        return None

    loan_id = "L" + str(len(loan_records) + 1).zfill(3)

    record = {
        "loan_id": loan_id,
        "member_id": member_id,
        "isbn": isbn,
        "branch_code": branch_code,
        "issue_date": issue_date,
        "due_date": "2026-06-15",
        "return_date": "",
        "days_overdue": 0,
        "fine_amount": 0,
        "fine_paid": False
    }

    loan_records.append(record)

    branch["active_loans"] += 1
    member["active_loans"].append(loan_id)

    return record



def process_return(loan_records,
                   members,
                   member_id,
                   isbn,
                   return_date):


    for loan in loan_records:

        if (loan["member_id"] == member_id and
                loan["isbn"] == isbn and
                loan["return_date"] == ""):

      
            days_overdue = 5

            member_type = members[member_id]["membership_type"]

            fine = calculate_fine(member_type,
                                  days_overdue,
                                  1,
                                  False)

        
            loan["return_date"] = return_date
            loan["days_overdue"] = days_overdue
            loan["fine_amount"] = fine

            members[member_id]["outstanding_fines"] += fine

            return fine

    return -1


def authenticate_staff(staff_accounts,
                       username,
                       password):
  
    if username not in staff_accounts:
        return None

    account = staff_accounts[username]

    if account["password"] == password:

        return {
            "role": account["role"],
            "full_name": account["full_name"]
        }

    return None

def generate_branch_report(loan_records,
                           members,
                           branch_code):
 
    total_loans = 0
    active_loans = 0
    overdue = 0
    fines_issued = 0
    fines_collected = 0
    total_duration = 0

    for loan in loan_records:

        if loan["branch_code"] == branch_code:

            total_loans += 1

            if loan["return_date"] == "":
                active_loans += 1

            if loan["days_overdue"] > 0:
                overdue += 1

            fines_issued += loan["fine_amount"]

            if loan["fine_paid"]:
                fines_collected += loan["fine_amount"]

            total_duration += 14

    if total_loans > 0:
        avg_duration = total_duration / total_loans
    else:
        avg_duration = 0

    return {
        "total_loans": total_loans,
        "active_loans": active_loans,
        "overdue": overdue,
        "fines_issued": fines_issued,
        "fines_collected": fines_collected,
        "avg_duration": avg_duration
    }


print("===== TESTING calculate_fine =====")

print(calculate_fine("Standard", 5, 2))
print(calculate_fine("Senior", 50, 2))


print("\n===== TESTING issue_loan =====")

result1 = issue_loan(
    loan_records,
    members,
    branches,
    "M100",
    "9781234567890",
    "2026-05-27"
)

print(result1)

result2 = issue_loan(
    loan_records,
    members,
    branches,
    "M101",
    "9781234567890",
    "2026-05-27"
)

print(result2)


print("\n===== TESTING process_return =====")

print(process_return(
    loan_records,
    members,
    "M100",
    "9781234567890",
    "2026-05-30"
))

print(process_return(
    loan_records,
    members,
    "M999",
    "9781234567890",
    "2026-05-30"
))


print("\n===== TESTING authenticate_staff =====")

print(authenticate_staff(
    staff_accounts,
    "admin1",
    "admin123"
))

print(authenticate_staff(
    staff_accounts,
    "admin1",
    "wrongpass"
))


print("\n===== TESTING generate_branch_report =====")

print(generate_branch_report(
    loan_records,
    members,
    "CEN"
))

print(generate_branch_report(
    loan_records,
    members,
    "NOR"
))

===== TESTING calculate_fine =====
25.0
60

===== TESTING issue_loan =====
{'loan_id': 'L001', 'member_id': 'M100', 'isbn': '9781234567890', 'branch_code': 'CEN', 'issue_date': '2026-05-27', 'due_date': '2026-06-15', 'return_date': '', 'days_overdue': 0, 'fine_amount': 0, 'fine_paid': False}
Branch loan capacity reached!
None

===== TESTING process_return =====
12.5
-1

===== TESTING authenticate_staff =====
{'role': 'Administrator', 'full_name': 'Alice Johnson'}
None

===== TESTING generate_branch_report =====
{'total_loans': 1, 'active_loans': 0, 'overdue': 1, 'fines_issued': 12.5, 'fines_collected': 0, 'avg_duration': 14.0}
{'total_loans': 0, 'active_loans': 0, 'overdue': 0, 'fines_issued': 0, 'fines_collected': 0, 'avg_duration': 0}


In [ ]:

def junior_fine(days_overdue, first_offence):


    if first_offence:
        return 0.0

    return days_overdue * 1.00


def standard_fine(days_overdue, num_books):


    return days_overdue * 2.50 * num_books


def senior_fine(days_overdue, num_books):


    fine_per_book = days_overdue * 1.00

    if fine_per_book > 30:
        fine_per_book = 30

    return fine_per_book * num_books


def student_fine(days_overdue, num_books):

    return days_overdue * 1.50 * num_books


def calculate_fine(membership_type,
                   days_overdue,
                   num_books,
                   first_offence=False):
   

    membership_type = membership_type.lower()

    if membership_type == "junior":
        return junior_fine(days_overdue,
                           first_offence) * num_books

    elif membership_type == "standard":
        return standard_fine(days_overdue,
                             num_books)

    elif membership_type == "senior":
        return senior_fine(days_overdue,
                           num_books)

    elif membership_type == "student":
        return student_fine(days_overdue,
                            num_books)

    return 0.0


print(calculate_fine("Junior", 5, 2, True))
print(calculate_fine("Junior", 5, 2, False))

print(calculate_fine("Standard", 4, 3))

print(calculate_fine("Senior", 50, 2))

print(calculate_fine("Student", 6, 2))

0.0
10.0
30.0
60
18.0
